## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions



In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq
model =ChatGroq(model="openai/gpt-oss-20b", groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000022FC939A210>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000022FC94F7020>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hello! My name is Anto.")])

AIMessage(content='Hello, Anto! 👋 How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "Hello! My name is Anto." This is a greeting. The user is introducing themselves. We should respond politely, greet them back, maybe ask how we can help. No special instructions. We\'ll respond accordingly.'}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 79, 'total_tokens': 150, 'completion_time': 0.07851203, 'completion_tokens_details': {'reasoning_tokens': 48}, 'prompt_time': 0.003854207, 'prompt_tokens_details': None, 'queue_time': 0.346201497, 'total_time': 0.082366237}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_996f667773', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0eb60-1116-7b41-bf4c-908603dbceb3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 79, 'output_tokens': 71, 'total_tokens': 150, 'output_token_details': {'

In [4]:
from langchain_core.messages import AIMessage 
model.invoke([
    HumanMessage(content="Hello! My name is Anto.I ma Ai engineer, developing AI products at enterprise level."),
    AIMessage(content="Hello Anto! 👋 How’s it going today? If you’ve got any questions or need a quick chat, I’m all ears."),
    HumanMessage(content="Hey Whats my name and what do i do?")
])

AIMessage(content='You’re Anto, and you’re an AI engineer building enterprise‑level AI products.', additional_kwargs={'reasoning_content': 'User: "Hey Whats my name and what do i do?" They previously said: "Hello! My name is Anto.I ma Ai engineer, developing AI products at enterprise level." So their name is Anto, they are an AI engineer developing AI products at enterprise level. So answer: your name is Anto, you are an AI engineer. Provide short.'}, response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 139, 'total_tokens': 241, 'completion_time': 0.109885858, 'completion_tokens_details': {'reasoning_tokens': 75}, 'prompt_time': 0.031108095, 'prompt_tokens_details': None, 'queue_time': 0.132037212, 'total_time': 0.140993953}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c5a89987dc', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0eb60-13bb-7272-89d6-0c89e877ec3f-0', tool_calls=[

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

C:\Users\antoc\AppData\Local\Temp\ipykernel_36028\1529080270.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory


In [6]:
store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id  not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history =RunnableWithMessageHistory(model, get_session_history)

d:\Udemy_Generative_AI_course\Enterprise_AI_Engineer_RoadMap\04-LangChain\venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [7]:
config={"configurable":{"session_id":"chat1"}}

In [8]:
response=with_message_history.invoke([
    HumanMessage(content="Hello! My name is Anto.I ma Ai engineer, developing AI products at enterprise level.")]
    , config=config)  

In [9]:
response.content

'Hi Anto! 👋  \nIt’s great to meet an AI engineer working on enterprise‑scale products. I’m here to help with anything from model design and deployment strategies to scaling, compliance, or even just brainstorming new features. What’s on your plate right now?'

In [10]:
with_message_history.invoke([
    HumanMessage(content="Hey Whats my name and what do i do?")]
    , config=config)

AIMessage(content='You’re Anto, and you’re an AI engineer building enterprise‑level AI products. 🚀', additional_kwargs={'reasoning_content': 'User says: "Hey Whats my name and what do i do?" They previously said "Hello! My name is Anto.I ma Ai engineer, developing AI products at enterprise level." So the user is Anto, they are an AI engineer developing AI products at enterprise level. The user is asking "Hey Whats my name and what do i do?" We need to answer: Their name is Anto and they are an AI engineer developing AI products at enterprise level. We should respond politely and confirm.'}, response_metadata={'token_usage': {'completion_tokens': 131, 'prompt_tokens': 165, 'total_tokens': 296, 'completion_time': 0.146183576, 'completion_tokens_details': {'reasoning_tokens': 102}, 'prompt_time': 0.007968889, 'prompt_tokens_details': None, 'queue_time': 0.308181614, 'total_time': 0.154152465}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_4f7e7dc26e', 'service_tier': 'on_d

In [11]:
config1={"configurable":{"session_id":"chat1"}}
response_1=with_message_history.invoke([
    HumanMessage(content="Hey Whats my name and what do i do?")]
    , config=config1)
response_1.content

'You’re Anto, and you’re an AI engineer developing AI products at the enterprise level.'

In [12]:
config1={"configurable":{"session_id":"chat2"}}
response_2=with_message_history.invoke([
    HumanMessage(content="Hey Whats my name and what do i do?")]
    , config=config1)
response_2.content

'I’m sorry, but I don’t have that information.'

In [13]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config1
)
response.content

'Nice to meet you, John! How can I help you today?'

In [14]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'You’re John. How can I help you today?'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [15]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful AI chatbot.Answer all the question to the best of your ability."),
        MessagesPlaceholder(variable_name="messages")
    ]
)
chain =prompt | model

In [16]:
chain.invoke({"messages":[
    HumanMessage(content="Hello! My name is Anto.I ma Ai engineer, developing AI products at enterprise level.")]})

AIMessage(content='Hello Anto! 👋 It’s great to meet an AI engineer working on enterprise‑level products. How can I assist you today? Whether you’re looking for insights on model deployment, best practices for data pipelines, or just a quick brainstorming session, I’m here to help.', additional_kwargs={'reasoning_content': 'User says "Hello! My name is Anto. I am AI engineer, developing AI products at enterprise level." They probably want a reply. We\'ll respond politely, maybe ask how we can help.'}, response_metadata={'token_usage': {'completion_tokens': 107, 'prompt_tokens': 111, 'total_tokens': 218, 'completion_time': 0.115460202, 'completion_tokens_details': {'reasoning_tokens': 41}, 'prompt_time': 0.051806928, 'prompt_tokens_details': None, 'queue_time': 0.416332635, 'total_time': 0.16726713}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_334cc21c60', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0

In [17]:
with_message_history =RunnableWithMessageHistory(chain, get_session_history)
config={"configurable":{"session_id":"chat3"}}

d:\Udemy_Generative_AI_course\Enterprise_AI_Engineer_RoadMap\04-LangChain\venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [18]:
response = with_message_history.invoke([
    HumanMessage(content="Hello! My name is Anto.I am Ai engineer, developing AI products at enterprise level.")],
    config=config

)

response.content

'Hi Anto! 👋 It’s great to meet an AI engineer—what kind of enterprise AI products are you working on right now? Whether it’s NLP, computer vision, recommendation systems, or something else, I’d love to hear more about your projects and any challenges you’re tackling.'

In [19]:
with_message_history.invoke([
    HumanMessage(content="Hey Whats my name and what do i do?")
],config=config)

AIMessage(content='You’re Anto, an AI engineer who’s building and deploying AI products at the enterprise level. In short, you design, develop, and scale AI solutions—whether that’s NLP, computer vision, recommendation engines, or other AI systems—to solve real business problems for large organizations.', additional_kwargs={'reasoning_content': 'User says: "Hey Whats my name and what do i do?" They previously said their name is Anto and they are an AI engineer developing AI products at enterprise level. So we respond: name is Anto, and you are an AI engineer developing enterprise AI products. Possibly mention what that entails.'}, response_metadata={'token_usage': {'completion_tokens': 128, 'prompt_tokens': 190, 'total_tokens': 318, 'completion_time': 0.141805334, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.013109935, 'prompt_tokens_details': None, 'queue_time': 0.312000551, 'total_time': 0.154915269}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint'

In [20]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [21]:
response = chain.invoke(
    {
        "messages": [
            HumanMessage(content="What is generate AI?"),
        ],
        "language": "Tamil",
    }
)
response.content

'**ஜெனரேட்டிவ் AI (Generative AI)** என்பது புதிய உள்ளடக்கங்களை உருவாக்குவதற்காக வடிவமைக்கப்பட்ட செயற்கை நுண்ணறிவு (Artificial Intelligence) தொழில்நுட்பம். இது முன்பு பயிற்சி பெற்ற தரவுகளின் முறை, வடிவமைப்பு, மற்றும் பாணியை கற்றுக்கொண்டு, அதனை அடிப்படையாகக் கொண்டு புதிய உரைகள், படங்கள், வீடியோக்கள், இசை, குறியீடு போன்றவற்றை உருவாக்க முடியும்.\n\n---\n\n## 1. எப்படி செயல்படுகிறது?\n\n| படி | விளக்கம் |\n|------|----------|\n| **தரவு சேகரிப்பு** | மிகப்பெரிய அளவு உரை, படங்கள், ஒலி போன்ற தரவுகள் சேகரிக்கப்படுகின்றன. |\n| **மாதிரி பயிற்சி** | இத்தரவைப் பயன்படுத்தி மெஷின் லேர்னிங் மாதிரிகள் (பொதுவாக டிரான்ஸ்ஃபார்மர், GANs, Diffusion Models) பயிற்றுவிக்கப்படுகின்றன. |\n| **கற்றல்** | மாதிரி தரவுகளில் உள்ள முறை, இலக்கணம், பாணி ஆகியவற்றை கற்றுக்கொள்கிறது. |\n| **உருவாக்கம்** | பயனர் வழங்கும் "prompt" (உதாரணத்திற்கு, ஒரு தலைப்பு அல்லது குறுகிய விளக்கம்) அடிப்படையில் மாதிரி புதிய உள்ளடக்கத்தை உருவாக்குகிறது. |\n\n---\n\n## 2. முக்கிய வகைகள்\n\n| வகை | உதாரணம் | பயன்பாடு |\n|------|---------|-----

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [22]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history, input_messages_key="messages")

d:\Udemy_Generative_AI_course\Enterprise_AI_Engineer_RoadMap\04-LangChain\venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [23]:
config={"configurable":{"session_id":"chat4"}}
response = with_message_history.invoke(
    {
        "messages": [
            HumanMessage(content="Hi my name is Anto"),
        ],
        "language": "Tamil",
    },
    config=config
)
response.content

'வணக்கம்,\u202fஅன்தோ! உங்களுக்கு எப்படி உதவலாம்?'

In [24]:
response = with_message_history.invoke(
    {
        "messages": [
            HumanMessage(content="What is my name?"),
        ],
        "language": "Tamil",
    },
    config=config
)
response.content

'உங்கள் பெயர் **அன்தோ**.'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [25]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer = trim_messages(
    max_tokens=70,
    strategy="last",
    token_counter=model,
    include_system=True,
    start_on="human"
)
    

In [26]:
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

In [27]:
trimmer.invoke(messages)

d:\Udemy_Generative_AI_course\Enterprise_AI_Engineer_RoadMap\04-LangChain\venv\Lib\site-packages\langchain_core\language_models\base.py:463: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))
d:\Udemy_Generative_AI_course\Enterprise_AI_Engineer_RoadMap\04-LangChain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Udemy_Generative_AI_course\Enterprise_AI_Engineer_RoadMap\04-LangChain\venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\antoc\.cache\huggingface\hub\models--gpt2. Ca

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="hi! I'm bob", additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs

In [29]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain=(RunnablePassthrough(messages = itemgetter("messages") | trimmer)
       | prompt
       | model)

response=chain.invoke({"messages":messages + [HumanMessage(content="what is the ice cream do i like ?")],
                       "language":"Tamil"})
response.content

'நீங்கள் விரும்பும் ஐஸ் க்ரீம் **வனிலா** (Vanilla) தான்!'

In [30]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

d:\Udemy_Generative_AI_course\Enterprise_AI_Engineer_RoadMap\04-LangChain\venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [31]:
response = with_message_history.invoke(
    {
        "messages": [
            HumanMessage(content="Hi my name is Anto"),
        ],
        "language": "Tamil",
    },
    config=config
)
response.content

'வணக்கம், Anto! நான் உங்களுக்கு எப்படி உதவ முடியும்?'

In [32]:
response = with_message_history.invoke(
    {
        "messages": [
            HumanMessage(content="what's my name ?"),
        ],
        "language": "Tamil",
    },
    config=config
)
response.content

'உங்கள் பெயர் **Anto**.'